# OrientaBot · Proyecto Fase 1
**Materia:** Aplicaciones con Inteligencia Artificial  
**Estudiante:** [Escribe tu nombre]  
**Semana 2 · Ponderación: 15 %**

Asistente escolar basado en reglas con Python y NLTK. Información demostrativa: no contiene fechas ni requisitos oficiales.

## Cómo usar este notebook
1. Ejecuta las celdas en orden con el botón ▶ o **Entorno de ejecución → Ejecutar todas**.
2. Revisa las diez conversaciones automáticas y descarga sus evidencias en la última celda.
3. Activa la conversación manual cuando quieras conversar con el asistente.
4. Guarda una copia en Drive. Para GitHub usa **Archivo → Guardar una copia en GitHub** y selecciona tu repositorio. Ese paso requiere tu cuenta.

Las pruebas automáticas son conversaciones simuladas; complementa sus registros con capturas de tu ejecución real en Colab.

## Taller práctico de una hora

| Minutos | Actividad |
|---|---|
| 0–10 | Explicar el problema, las intenciones y la arquitectura. |
| 10–20 | Abrir Colab, instalar NLTK y ejecutar el chatbot. |
| 20–35 | Activar el chat manual y probar consultas. |
| 35–45 | Agregar una intención propia antes de la regla desconocida y volver a ejecutar la base de conocimiento. |
| 45–55 | Ejecutar las diez conversaciones y revisar resultados. |
| 55–60 | Identificar limitaciones y guardar el notebook. |

**Para después de clase:** completar el PDF de 8 a 10 cuartillas, guardar el código en GitHub y agregar capturas de ejecución.

**Reto:** añade una intención sobre biblioteca antes de `desconocida`, con el patrón `r".*\b(biblioteca|libros|prestamo)\b.*"` y una respuesta general que remita al área responsable. Después vuelve a ejecutar la celda y prueba «Quiero pedir libros». No inventes horarios ni requisitos oficiales.

## 1. Definición de la solución
**Problema:** los estudiantes necesitan orientación inicial sobre trámites escolares frecuentes y las áreas responsables.

**Objetivo general:** implementar un chatbot que responda consultas sobre inscripciones, horarios, becas y constancias mediante reglas predefinidas.

**Usuarios:** estudiantes de una institución educativa.

**Alcance:** orientación general. No realiza trámites, consulta expedientes ni confirma convocatorias.

**Beneficios esperados:** disponibilidad de una primera orientación y respuestas consistentes. Estos beneficios requieren evaluación posterior con usuarios reales.

## 2. Arquitectura conceptual
Usuario → entrada de texto → normalización → motor de reglas NLTK ↔ base de conocimiento → respuesta → usuario.

El flujo comienza con una consulta; se identifica el primer patrón coincidente y se devuelve su respuesta. Una regla final atiende consultas desconocidas. No se conserva contexto entre turnos.

In [ ]:
import sys
assert sys.version_info >= (3, 10), 'Se requiere Python 3.10 o superior.'
%pip -q install nltk==3.9.1
print('Python:', sys.version.split()[0])

## 3. Base de conocimiento y diseño conversacional
Cada intención contiene un identificador, un patrón y una respuesta. El orden importa: se utiliza la primera coincidencia. No se requieren corpus adicionales de NLTK para este motor de reglas.

In [ ]:
import re
import unicodedata
import nltk
from nltk.chat.util import Chat, reflections
from IPython.display import display, Markdown

BASE = [
    ('saludo', r'^(hola|buenos dias|buenas tardes|buenas noches|hey)\b.*', '¡Hola! Soy OrientaBot. Consulta sobre inscripciones, horarios, becas o constancias.'),
    ('inscripcion', r'.*\b(inscripcion|inscripciones|inscribirme|reinscripcion)\b.*', 'Consulta la convocatoria vigente y contacta a Control Escolar para conocer el proceso de inscripción o reinscripción. Verifica allí fechas y requisitos oficiales.'),
    ('horarios', r'.*\b(horario|horarios|clases)\b.*', 'Consulta tu horario en el sistema escolar institucional. Si existe algún problema con tu grupo o materia, contacta a Control Escolar.'),
    ('becas', r'.*\b(beca|becas|apoyo economico)\b.*', 'Consulta las convocatorias institucionales vigentes. El área de Becas puede orientarte sobre requisitos, documentos y fechas.'),
    ('constancias', r'.*\b(constancia|constancias|comprobante de estudios)\b.*', 'Solicita información a Control Escolar sobre el procedimiento, documentos necesarios y tiempo de entrega de constancias.'),
    ('control_escolar', r'.*\b(control escolar|servicios escolares)\b.*', 'Control Escolar orienta sobre trámites académicos. Consulta sus medios de contacto en el sitio oficial de tu institución.'),
    ('ayuda', r'.*\b(ayuda|menu|opciones|que puedes hacer)\b.*', 'Temas disponibles: inscripciones, horarios, becas, constancias y Control Escolar. Escribe salir para terminar.'),
    ('agradecimiento', r'.*\bgracias\b.*', '¡De nada! ¿Necesitas orientación sobre otro tema?'),
    ('despedida', r'^(salir|adios|hasta luego)[.!?]*$', '¡Hasta luego! Gracias por consultar OrientaBot.'),
    ('vacio', r'^$', 'Escribe una consulta para poder orientarte.'),
    ('desconocida', r'.*', 'No identifiqué tu consulta. Intenta mencionar inscripciones, horarios, becas, constancias o Control Escolar.')
]

chatbot = Chat([(patron, [respuesta]) for _, patron, respuesta in BASE], reflections)

def normalizar(texto):
    texto = unicodedata.normalize('NFD', texto.lower().strip())
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', texto)

def responder(texto):
    limpio = normalizar(texto)
    intencion = next(nombre for nombre, patron, _ in BASE if re.match(patron, limpio, re.IGNORECASE))
    return {'intencion': intencion, 'respuesta': chatbot.respond(limpio)}

for nombre, patron, respuesta in BASE:
    print(f'{nombre}: {respuesta}')
print('Versión NLTK:', nltk.__version__)

## 4. Diez conversaciones distintas de prueba
Incluyen saludo, trámites, acentos, mayúsculas, texto vacío, una consulta fuera de alcance y una consulta con dos temas. Esta última muestra la prioridad de reglas; no demuestra comprensión de ambas intenciones.

Las intenciones esperadas se declaran antes de ejecutar las pruebas. La tasa obtenida describe únicamente estos casos diseñados, no la precisión general ante usuarios reales.

In [ ]:
CONVERSACIONES = [
    [('Hola', 'saludo'), ('¿Cómo puedo inscribirme?', 'inscripcion'), ('Gracias', 'agradecimiento')],
    [('Buenas tardes', 'saludo'), ('¿Dónde veo mis horarios?', 'horarios')],
    [('Necesito una beca', 'becas'), ('Adiós', 'despedida')],
    [('Quiero una constancia', 'constancias'), ('Gracias', 'agradecimiento')],
    [('¿Dónde está Control Escolar?', 'control_escolar'), ('salir', 'despedida')],
    [('¿Qué puedes hacer?', 'ayuda'), ('¿Cómo es la reinscripción?', 'inscripcion')],
    [('HOLA', 'saludo'), ('BUSCO APOYO ECONÓMICO', 'becas')],
    [('   ', 'vacio'), ('Necesito mi comprobante de estudios', 'constancias')],
    [('¿Qué clima hará mañana?', 'desconocida'), ('menú', 'ayuda')],
    [('Quiero saber de becas y horarios', 'horarios'), ('hasta luego', 'despedida')]
]
registros = []
for numero, conversacion in enumerate(CONVERSACIONES, 1):
    print(f'\n=== Conversación {numero:02d} ===')
    for entrada, esperada in conversacion:
        salida = responder(entrada)
        correcto = salida['intencion'] == esperada
        registros.append({'conversacion': numero, 'entrada': entrada, 'esperada': esperada, **salida, 'correcto': correcto})
        print('Usuario:', repr(entrada))
        print('OrientaBot:', salida['respuesta'])
        print('Intención:', salida['intencion'], '| Resultado:', 'OK' if correcto else 'REVISAR')

aciertos = sum(r['correcto'] for r in registros)
print(f'\nResultado: {aciertos}/{len(registros)} turnos coinciden con lo esperado.')
assert aciertos == len(registros), 'Hay casos que requieren revisión.'

## 5. Conversación manual opcional
Cambia `ACTIVAR_CHAT` a `True` y ejecuta la celda. Escribe `salir` para terminar. Se deja desactivada para que **Ejecutar todas** no quede esperando una entrada.

In [ ]:
ACTIVAR_CHAT = False
if ACTIVAR_CHAT:
    print('OrientaBot: ¡Hola! Escribe tu consulta o salir.')
    while True:
        entrada = input('Tú: ')
        salida = responder(entrada)
        print('OrientaBot:', salida['respuesta'])
        if salida['intencion'] == 'despedida':
            break
else:
    print('Activa ACTIVAR_CHAT para conversar manualmente.')

## 6. Exportación de evidencias
Genera un archivo CSV y uno de texto con las respuestas obtenidas en esta ejecución y los empaqueta en ZIP. En Colab puedes activar su descarga. Captura además las salidas visibles del notebook para el PDF.

In [ ]:
import csv
import json
import zipfile
from pathlib import Path
from datetime import datetime, timezone

carpeta = Path('evidencias_orientabot')
carpeta.mkdir(exist_ok=True)
with (carpeta / 'pruebas.csv').open('w', newline='', encoding='utf-8-sig') as f:
    escritor = csv.DictWriter(f, fieldnames=list(registros[0]))
    escritor.writeheader()
    escritor.writerows(registros)
lineas = ['Ejecución UTC: ' + datetime.now(timezone.utc).isoformat(), f'Python: {sys.version.split()[0]} | NLTK: {nltk.__version__}']
for r in registros:
    lineas.extend([f"Conversación {r['conversacion']} | Usuario: {r['entrada']!r}", 'OrientaBot: ' + r['respuesta'], f"Intención: {r['intencion']} | Esperada: {r['esperada']} | Correcto: {r['correcto']}", ''])
(carpeta / 'conversaciones.txt').write_text('\n'.join(lineas), encoding='utf-8')
with zipfile.ZipFile('Evidencias_OrientaBot.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for archivo in carpeta.iterdir():
        z.write(archivo, arcname=archivo.name)
print('Evidencias generadas: Evidencias_OrientaBot.zip')
DESCARGAR = False  # Cambia a True si deseas descargar el ZIP en Colab.
if DESCARGAR:
    from google.colab import files
    files.download('Evidencias_OrientaBot.zip')

## 7. Resultados, limitaciones y siguientes etapas
Completa el análisis después de ejecutar:
- ¿Cuántos turnos coincidieron con la intención esperada?
- ¿Qué ocurrió con la consulta sobre el clima y el texto vacío?
- ¿Por qué se respondió sobre horarios ante la consulta con dos temas?
- Prueba «quiero inscribirme» y después «¿qué documentos necesito?». ¿Conserva el tema?

**Limitaciones:** coincidencias por palabras; prioridad por orden; no conserva contexto; no maneja negaciones ni errores ortográficos; información general que requiere validación institucional. Las diez conversaciones son pruebas controladas, no una evaluación representativa.

**Mejoras:** ampliar el conjunto de pruebas con usuarios reales, incorporar clasificación de intenciones y manejo de contexto, validar la base de conocimiento e integrar posteriormente una interfaz web y clasificación de imágenes según el alcance elegido.

## 8. GitHub y entrega
Guarda una copia del notebook en GitHub desde el menú Archivo. Agrega el enlace real del repositorio al PDF. Este notebook no publica automáticamente ningún repositorio.

El PDF final de 8 a 10 cuartillas debe incluir portada, introducción, problema, objetivo, arquitectura, diseño conversacional, código fuente, evidencias, resultados, conclusiones y referencias.

## Referencias técnicas para consulta
- NLTK: https://www.nltk.org/api/nltk.chat.util.html
- Google Colab: https://colab.research.google.com/
- Python: https://docs.python.org/3/library/re.html